# Genie Conversation History Ingestion
Fetches all conversations and messages from a Genie Space via the Databricks REST API and writes them to a Unity Catalog Delta table.  
Supports **incremental updates**: on each run it only fetches conversations updated since the last run.

In [ ]:
dbutils.widgets.text("space_id", "", "Genie Space ID")
dbutils.widgets.text("catalog",  "",              "UC Catalog")
dbutils.widgets.text("schema",   "",                "UC Schema")
dbutils.widgets.text("table",    "", "UC Table")

space_id   = dbutils.widgets.get("space_id")
catalog    = dbutils.widgets.get("catalog")
schema     = dbutils.widgets.get("schema")
table      = dbutils.widgets.get("table")
full_table = f"`{catalog}`.`{schema}`.`{table}`"

print(f"Target table : {full_table}")
print(f"Genie space  : {space_id}")

## Setup

In [ ]:
import requests, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)

workspace_url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
token = (
    dbutils.notebook.entry_point
    .getDbutils().notebook().getContext()
    .apiToken().get()
)
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
print(f"Workspace: {workspace_url}")

## API helpers

In [ ]:
def _get(url, params=None):
    """GET with raise_for_status and JSON return."""
    r = requests.get(url, headers=headers, params=params or {}, timeout=30)
    r.raise_for_status()
    return r.json()


def list_conversations(space_id, page_token=None):
    """Paginated list of all conversations in a space.
    Returns (conversations_list, next_page_token_or_None).
    """
    params = {}
    if page_token:
        params["page_token"] = page_token
    data = _get(
        f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations",
        params=params,
    )
    return data.get("conversations", []), data.get("next_page_token")


def list_messages(space_id, conversation_id, page_token=None):
    """Paginated list of messages in a conversation.
    Returns (messages_list, next_page_token_or_None).
    """
    params = {}
    if page_token:
        params["page_token"] = page_token
    data = _get(
        f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages",
        params=params,
    )
    return data.get("messages", []), data.get("next_page_token")

## Incremental watermark

In [ ]:
# Determine the last-seen updated_at timestamp from the existing table.
# If the table doesn't exist yet, watermark = None (full load).

def get_watermark(full_table: str):
    try:
        row = spark.sql(
            f"SELECT MAX(conversation_updated_at) AS wm FROM {full_table}"
        ).collect()[0]
        return row["wm"]  # datetime or None
    except Exception:
        return None


watermark = get_watermark(full_table)
if watermark:
    print(f"Incremental load — fetching conversations updated after: {watermark}")
else:
    print("Full load — table does not exist or is empty.")

## Fetch conversations

In [ ]:
def fetch_all_conversations(space_id, watermark=None):
    """Yield all conversations, optionally filtered by updated_at > watermark."""
    page_token = None
    total = 0
    while True:
        convs, page_token = list_conversations(space_id, page_token)
        for c in convs:
            updated_ms = c.get("updated_at") or c.get("create_time") or 0
            updated_dt = datetime.fromtimestamp(updated_ms / 1000, tz=timezone.utc)
            # Skip conversations not updated since the watermark
            if watermark and updated_dt.replace(tzinfo=None) <= watermark.replace(tzinfo=None):
                continue
            yield c
            total += 1
        if not page_token:
            break
    print(f"Conversations to process: {total}")


conversations = list(fetch_all_conversations(space_id, watermark))
print(f"Fetched {len(conversations)} conversation(s)")

## Fetch messages

In [ ]:
def fetch_all_messages(space_id, conversation_id):
    """Return all messages for a single conversation (handles pagination)."""
    page_token = None
    msgs = []
    while True:
        batch, page_token = list_messages(space_id, conversation_id, page_token)
        msgs.extend(batch)
        if not page_token:
            break
    return msgs


records = []
for conv in conversations:
    cid          = conv.get("id") or conv.get("conversation_id", "")
    title        = conv.get("title", "")
    c_created_ms = conv.get("create_time") or conv.get("created_at") or 0
    c_updated_ms = conv.get("updated_at") or c_created_ms
    c_created    = datetime.fromtimestamp(c_created_ms / 1000, tz=timezone.utc)
    c_updated    = datetime.fromtimestamp(c_updated_ms / 1000, tz=timezone.utc)

    try:
        messages = fetch_all_messages(space_id, cid)
    except Exception as e:
        print(f"  WARNING: could not fetch messages for conversation {cid}: {e}")
        messages = []

    for msg in messages:
        feedback = msg.get("feedback") or {}
        records.append({
            "space_id":                space_id,
            "conversation_id":         cid,
            "conversation_title":      title,
            "conversation_created_at": c_created.replace(tzinfo=None),
            "conversation_updated_at": c_updated.replace(tzinfo=None),
            "message_id":              str(msg.get("id", "")),
            "message_type":            msg.get("author_user_id", {}).get("type", msg.get("type", "")),
            "message_content":         msg.get("content") or msg.get("text") or "",
            "message_status":          msg.get("status", ""),
            "message_created_at":      datetime.fromtimestamp(
                                           (msg.get("create_time") or 0) / 1000,
                                           tz=timezone.utc
                                       ).replace(tzinfo=None),
            "feedback_rating":         feedback.get("rating") if isinstance(feedback, dict) else str(feedback),
            "ingested_at":             datetime.utcnow(),
        })

print(f"Total message records collected: {len(records)}")

## Build DataFrame

In [ ]:
schema_def = StructType([
    StructField("space_id",                  StringType(),    False),
    StructField("conversation_id",           StringType(),    False),
    StructField("conversation_title",        StringType(),    True),
    StructField("conversation_created_at",   TimestampType(), True),
    StructField("conversation_updated_at",   TimestampType(), True),
    StructField("message_id",                StringType(),    False),
    StructField("message_type",              StringType(),    True),
    StructField("message_content",           StringType(),    True),
    StructField("message_status",            StringType(),    True),
    StructField("message_created_at",        TimestampType(), True),
    StructField("feedback_rating",           StringType(),    True),
    StructField("ingested_at",               TimestampType(), False),
])

if records:
    df_new = spark.createDataFrame(records, schema=schema_def)
else:
    df_new = spark.createDataFrame([], schema=schema_def)

print(f"New rows: {df_new.count()}")
df_new.printSchema()

## Write to Unity Catalog (MERGE)

In [ ]:
from delta.tables import DeltaTable

# Ensure catalog + schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

if df_new.isEmpty():
    print("Nothing new to write.")
else:
    # Create table on first run
    if not spark.catalog.tableExists(f"{catalog}.{schema}.{table}"):
        (
            df_new.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{catalog}.{schema}.{table}")
        )
        print(f"Table created and {df_new.count()} rows inserted.")
    else:
        dt = DeltaTable.forName(spark, f"{catalog}.{schema}.{table}")
        (
            dt.alias("tgt")
            .merge(
                df_new.alias("src"),
                "tgt.space_id = src.space_id "
                "AND tgt.conversation_id = src.conversation_id "
                "AND tgt.message_id = src.message_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("MERGE complete.")

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {full_table}").show()

## Popular questions (top 20)

In [ ]:
%sql

SELECT
    message_content                                AS question,
    COUNT(*)                                       AS times_asked,
    COUNT(DISTINCT conversation_id)                AS unique_conversations,
    SUM(CASE WHEN feedback_rating = 'THUMBS_UP'   THEN 1 ELSE 0 END) AS thumbs_up,
    SUM(CASE WHEN feedback_rating = 'THUMBS_DOWN' THEN 1 ELSE 0 END) AS thumbs_down,
    MIN(message_created_at)                        AS first_asked,
    MAX(message_created_at)                        AS last_asked
FROM IDENTIFIER(:catalog || '.' || :schema || '.' || :table)
WHERE message_type NOT IN ('GENIE', 'AI', 'ASSISTANT')   -- user messages only
  AND message_content IS NOT NULL
  AND LENGTH(TRIM(message_content)) > 0
GROUP BY message_content
ORDER BY times_asked DESC
LIMIT 20


## AI-powered topic clustering
Uses `ai_gen()` to normalise each distinct user question to a short canonical topic label, then groups by that label to surface the most-asked themes — including differently-phrased questions that share the same intent.

In [ ]:
%sql
-- ai_gen() is called once per DISTINCT question (not per row) — O(unique phrasings).
-- Each question is normalised to a 3-6 word canonical topic label so that
-- semantically equivalent questions (e.g. "show total revenue" vs "what is total revenue")
-- collapse into the same cluster.


WITH user_questions AS (
  SELECT
    message_content                                            AS question,
    COUNT(*)                                                   AS times_asked,
    COUNT(DISTINCT conversation_id)                            AS unique_conversations,
    SUM(CASE WHEN feedback_rating = 'THUMBS_UP'   THEN 1 ELSE 0 END) AS thumbs_up,
    SUM(CASE WHEN feedback_rating = 'THUMBS_DOWN' THEN 1 ELSE 0 END) AS thumbs_down,
    MIN(message_created_at)                                    AS first_asked,
    MAX(message_created_at)                                    AS last_asked
  FROM IDENTIFIER(:catalog || '.' || :schema || '.' || :table)
  WHERE message_type NOT IN ('GENIE', 'AI', 'ASSISTANT')
    AND message_content IS NOT NULL
    AND LENGTH(TRIM(message_content)) > 0
  GROUP BY message_content
),
normalized AS (
  SELECT
    question,
    times_asked,
    unique_conversations,
    thumbs_up,
    thumbs_down,
    first_asked,
    last_asked,
    LOWER(TRIM(
      ai_gen(
        'You are an analytics assistant. Normalise the following user question to a '
        || 'concise topic label of 3-6 words that captures its core business intent. '
        || 'Return ONLY the topic label — no punctuation, no explanation. '
        || 'Examples: "sales by region", "top customers by revenue", '
        || '"monthly active users", "product inventory levels". '
        || 'Question: ' || question
      )
    )) AS topic_label
  FROM user_questions
)
SELECT
  topic_label                                                  AS topic,
  SUM(times_asked)                                             AS total_times_asked,
  COUNT(*)                                                     AS unique_phrasings,
  SUM(unique_conversations)                                    AS total_conversations,
  SUM(thumbs_up)                                               AS total_thumbs_up,
  SUM(thumbs_down)                                             AS total_thumbs_down,
  MIN(first_asked)                                             AS first_asked,
  MAX(last_asked)                                              AS last_asked,
  FIRST(question)                                              AS example_question
FROM normalized
WHERE topic_label IS NOT NULL
  AND LENGTH(TRIM(topic_label)) > 0
GROUP BY topic_label
ORDER BY total_times_asked DESC
LIMIT 30
